In [ ]:
import pandas as pd
import arviz as az
import pycountry
import seaborn as sns
from matplotlib import pyplot as plt
from matplotlib_inline.backend_inline import set_matplotlib_formats
from matplotlib.pyplot import cm

from emu_renewal.inputs import OUTPUTS_PATH
from emu_renewal.outputs import get_country_likelihoods, get_country_disps
from emu_renewal.plotting import plot_like_comparison

In [ ]:
set_matplotlib_formats("svg")
colours = cm.Dark2.colors

In [ ]:
job_path = OUTPUTS_PATH / "42780561"
countries = [p.parts[-1] for p in job_path.iterdir()]

In [ ]:
#| fig-cap: "Comparison of posteriors by candidate mobility modifier."
likes = get_country_likelihoods(job_path, countries)
plot_like_comparison(likes, colours, "Posterior comparison", "like")

In [ ]:
#| fig-cap: "Comparison of dispersion values by candidate mobility modifier."
disps = get_country_disps(job_path, countries)
plot_like_comparison(disps, colours, "Dispersion comparison", "disp")

In [ ]:
procs = {}
for c in countries:
    country_path = job_path / c
    c_procs = []
    analyses = [a.parts[-1] for a in country_path.iterdir()]
    for a in analyses:
        analysis_path = country_path / a
        c_procs.append(pd.read_hdf(analysis_path / "spaghetti.h5")["process"])
    procs[c] = pd.concat(c_procs, keys=analyses, axis=1)

In [ ]:
#| fig-cap: "Comparison of variable process by candidate mobility modifier."
proc_fig, axes = plt.subplots(4, 4, figsize=[10, 10])
flat_axes = axes.ravel()
for c in range(len(countries)):
    country = countries[c]
    country_path = job_path / country
    analyses = [a.parts[-1] for a in country_path.iterdir()]
    for a, analysis in enumerate(analyses):
        data = procs[country][analysis].quantile([0.05, 0.5, 0.95], axis=1).T
        flat_axes[c].plot(data.index, data[0.5], color=colours[a], label=analysis, linewidth=3.0)
        flat_axes[c].plot(data.index, data[[0.05, 0.95]], color=colours[a], linestyle="--", linewidth=0.5)
        flat_axes[c].set_title(country)
        flat_axes[c].legend()
        flat_axes[c].set_yscale("log")
        plt.setp(flat_axes[c].xaxis.get_majorticklabels(), rotation=70)
    if c != 0:
        flat_axes[c].get_legend().remove()
proc_fig.savefig("proc_fig.svg")
proc_fig.tight_layout()

In [ ]:
#| tbl-cap: "Median likelihood values by country and candidate mobility modifier."
like_medians = pd.DataFrame()
for country in countries:
    like_medians[country] = likes[country].stack(level=1, future_stack=True).median()
like_medians.T